In [2]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Pranav\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [3]:
corpus = [
    "Natural Language Processing is a fascinating field of AI.",
    "Language models help machines understand human language.",
    "Bag of Words and TF-IDF are basic NLP techniques.",
    "Deep learning has improved NLP tasks significantly."
]

for i, doc in enumerate(corpus):
    print(f"Document {i+1}: {doc}")

Document 1: Natural Language Processing is a fascinating field of AI.
Document 2: Language models help machines understand human language.
Document 3: Bag of Words and TF-IDF are basic NLP techniques.
Document 4: Deep learning has improved NLP tasks significantly.


In [4]:
def preprocess(text):
    text = text.lower()                              # lowercase
    text = re.sub(r'[^a-z\s]', '', text)              # remove punctuation/numbers
    tokens = text.split()                             # tokenize
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words]  # remove stopwords
    ps = PorterStemmer()
    tokens = [ps.stem(word) for word in tokens]       # stemming
    return " ".join(tokens)

cleaned_corpus = [preprocess(doc) for doc in corpus]

for i, doc in enumerate(cleaned_corpus):
    print(f"Cleaned Document {i+1}: {doc}")

Cleaned Document 1: natur languag process fascin field ai
Cleaned Document 2: languag model help machin understand human languag
Cleaned Document 3: bag word tfidf basic nlp techniqu
Cleaned Document 4: deep learn improv nlp task significantli


In [5]:
# Using CountVectorizer to create Bag of Words
bow_vectorizer = CountVectorizer()
bow_matrix = bow_vectorizer.fit_transform(cleaned_corpus)

# Convert to DataFrame for readability
bow_df = pd.DataFrame(
    bow_matrix.toarray(),
    columns=bow_vectorizer.get_feature_names_out(),
    index=[f"Doc {i+1}" for i in range(len(corpus))]
)

print("Bag of Words Representation:\n")
bow_df

Bag of Words Representation:



,ai,bag,basic,deep,fascin,field,help,human,improv,languag,...,model,natur,nlp,process,significantli,task,techniqu,tfidf,understand,word
Doc 1,1,0,0,0,1,1,0,0,0,1,...,0,1,0,1,0,0,0,0,0,0
Doc 2,0,0,0,0,0,0,1,1,0,2,...,1,0,0,0,0,0,0,0,1,0
Doc 3,0,1,1,0,0,0,0,0,0,0,...,0,0,1,0,0,0,1,1,0,1
Doc 4,0,0,0,1,0,0,0,0,1,0,...,0,0,1,0,1,1,0,0,0,0


In [6]:
def manual_bow(corpus):
    # Build vocabulary
    vocab = sorted(set(word for doc in corpus for word in doc.split()))
    
    # Build BoW matrix
    bow_matrix = []
    for doc in corpus:
        words = doc.split()
        row = [words.count(word) for word in vocab]
        bow_matrix.append(row)
    
    return pd.DataFrame(bow_matrix, columns=vocab, index=[f"Doc {i+1}" for i in range(len(corpus))])

manual_bow_df = manual_bow(cleaned_corpus)
print("Manual Bag of Words:\n")
manual_bow_df

Manual Bag of Words:



,ai,bag,basic,deep,fascin,field,help,human,improv,languag,...,model,natur,nlp,process,significantli,task,techniqu,tfidf,understand,word
Doc 1,1,0,0,0,1,1,0,0,0,1,...,0,1,0,1,0,0,0,0,0,0
Doc 2,0,0,0,0,0,0,1,1,0,2,...,1,0,0,0,0,0,0,0,1,0
Doc 3,0,1,1,0,0,0,0,0,0,0,...,0,0,1,0,0,0,1,1,0,1
Doc 4,0,0,0,1,0,0,0,0,1,0,...,0,0,1,0,1,1,0,0,0,0


In [7]:
# Using TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(cleaned_corpus)

# Convert to DataFrame
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=tfidf_vectorizer.get_feature_names_out(),
    index=[f"Doc {i+1}" for i in range(len(corpus))]
)

print("TF-IDF Representation:\n")
tfidf_df.round(3)

TF-IDF Representation:



,ai,bag,basic,deep,fascin,field,help,human,improv,languag,...,model,natur,nlp,process,significantli,task,techniqu,tfidf,understand,word
Doc 1,0.422,0.000,0.000,0.000,0.422,0.422,0.000,0.000,0.000,0.333,...,0.000,0.422,0.000,0.422,0.000,0.000,0.000,0.000,0.000,0.000
Doc 2,0.000,0.000,0.000,0.000,0.000,0.000,0.365,0.365,0.000,0.576,...,0.365,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.365,0.000
Doc 3,0.000,0.422,0.422,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.333,0.000,0.000,0.000,0.422,0.422,0.000,0.422
Doc 4,0.000,0.000,0.000,0.422,0.000,0.000,0.000,0.000,0.422,0.000,...,0.000,0.000,0.333,0.000,0.422,0.422,0.000,0.000,0.000,0.000


In [8]:
import math

def compute_tf(doc):
    words = doc.split()
    word_count = len(words)
    tf_dict = {}
    for word in words:
        tf_dict[word] = tf_dict.get(word, 0) + 1
    for word in tf_dict:
        tf_dict[word] = tf_dict[word] / word_count
    return tf_dict

def compute_idf(corpus):
    N = len(corpus)
    all_words = set(word for doc in corpus for word in doc.split())
    idf_dict = {}
    for word in all_words:
        containing_docs = sum(1 for doc in corpus if word in doc.split())
        idf_dict[word] = math.log((N) / (containing_docs)) + 1  # smoothed
    return idf_dict

def compute_tfidf(corpus):
    idf_dict = compute_idf(corpus)
    tfidf_list = []
    for doc in corpus:
        tf_dict = compute_tf(doc)
        tfidf = {word: tf_dict[word] * idf_dict[word] for word in tf_dict}
        tfidf_list.append(tfidf)
    return tfidf_list

manual_tfidf = compute_tfidf(cleaned_corpus)

# Display nicely
all_vocab = sorted(set(word for doc in cleaned_corpus for word in doc.split()))
manual_tfidf_df = pd.DataFrame(manual_tfidf, index=[f"Doc {i+1}" for i in range(len(corpus))])
manual_tfidf_df = manual_tfidf_df[all_vocab].fillna(0)
print("Manual TF-IDF Representation:\n")
manual_tfidf_df.round(3)

Manual TF-IDF Representation:



,ai,bag,basic,deep,fascin,field,help,human,improv,languag,...,model,natur,nlp,process,significantli,task,techniqu,tfidf,understand,word
Doc 1,0.398,0.000,0.000,0.000,0.398,0.398,0.000,0.000,0.000,0.282,...,0.000,0.398,0.000,0.398,0.000,0.000,0.000,0.000,0.000,0.000
Doc 2,0.000,0.000,0.000,0.000,0.000,0.000,0.341,0.341,0.000,0.484,...,0.341,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.341,0.000
Doc 3,0.000,0.398,0.398,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.282,0.000,0.000,0.000,0.398,0.398,0.000,0.398
Doc 4,0.000,0.000,0.000,0.398,0.000,0.000,0.000,0.000,0.398,0.000,...,0.000,0.000,0.282,0.000,0.398,0.398,0.000,0.000,0.000,0.000


In [9]:
print("Top word per document (BoW - highest frequency):")
for i in range(len(corpus)):
    top_word = bow_df.iloc[i].idxmax()
    print(f"Doc {i+1}: '{top_word}' -> count = {bow_df.iloc[i].max()}")

print("\nTop word per document (TF-IDF - highest importance):")
for i in range(len(corpus)):
    top_word = tfidf_df.iloc[i].idxmax()
    print(f"Doc {i+1}: '{top_word}' -> score = {tfidf_df.iloc[i].max():.3f}")

Top word per document (BoW - highest frequency):
Doc 1: 'ai' -> count = 1
Doc 2: 'languag' -> count = 2
Doc 3: 'bag' -> count = 1
Doc 4: 'deep' -> count = 1

Top word per document (TF-IDF - highest importance):
Doc 1: 'ai' -> score = 0.422
Doc 2: 'languag' -> score = 0.576
Doc 3: 'bag' -> score = 0.422
Doc 4: 'deep' -> score = 0.422


In [10]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(tfidf_matrix)
similarity_df = pd.DataFrame(
    similarity_matrix,
    index=[f"Doc {i+1}" for i in range(len(corpus))],
    columns=[f"Doc {i+1}" for i in range(len(corpus))]
)

print("Cosine Similarity Matrix (based on TF-IDF):\n")
similarity_df.round(3)

Cosine Similarity Matrix (based on TF-IDF):



,Doc 1,Doc 2,Doc 3,Doc 4
Doc 1,1.000,0.192,0.000,0.000
Doc 2,0.192,1.000,0.000,0.000
Doc 3,0.000,0.000,1.000,0.111
Doc 4,0.000,0.000,0.111,1.000
